# Border-03-TinyML-Training
## Edge AI-Based Acoustic Footstep Detection for Border Surveillance
### Notebook 03 · Model Training, Quantisation & Deployment Profiling

**Pipeline position:** EDA → Preprocessing → **Training ← (you are here)** → Deployment

**Inputs:** `features_train.npz`, `features_val.npz`, `features_test.npz`  
**Outputs:** `border_model.tflite` (INT8), `border_model_float.tflite`, evaluation figures, classification report

**TinyML target:** ESP32 (240 MHz, ~320 KB SRAM, ~4 MB Flash)

In [7]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import warnings, os, time, itertools, json, struct
warnings.filterwarnings("ignore")

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from tensorflow.keras.utils import to_categorical
print(f"TensorFlow {tf.__version__}  |  Keras {keras.__version__}")

# Sklearn metrics
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
INPUT_PATH   = "/kaggle/input/borderintrusiondetection-data"
WORKING_PATH1 = "/kaggle/input/datasets/katakuricharlotte/border-preprocessed-data/processed"
WORKING_PATH ="/kaggle/working/"
FIGURES_PATH = "/kaggle/working/figures"
os.makedirs(FIGURES_PATH, exist_ok=True)

# ── Paper-ready plot styling ───────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "#F8F9FA",
    "axes.edgecolor": "#CCCCCC",
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.color": "#FFFFFF",
    "grid.linewidth": 1.2,
    "grid.alpha": 0.7,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "#CCCCCC",
    "lines.linewidth": 1.8,
    "patch.linewidth": 0.5,
    "savefig.bbox": "tight",
    "savefig.dpi": 300,
    "savefig.facecolor": "white",
})

CLASS_COLORS = {"balastic": "#2196F3", "footsteps": "#FF9800", "noise": "#4CAF50"}
CLASS_NAMES  = ["balastic", "footsteps", "noise"]
PALETTE      = [CLASS_COLORS[c] for c in CLASS_NAMES]

def paper_axes(ax):
    ax.set_facecolor("#F8F9FA")
    for spine in ax.spines.values():
        spine.set_edgecolor("#CCCCCC")
        spine.set_linewidth(0.8)
    ax.grid(True, color="white", linewidth=1.2, alpha=0.7)
    ax.tick_params(colors="#444444")
    ax.title.set_color("#222222")
    ax.xaxis.label.set_color("#444444")
    ax.yaxis.label.set_color("#444444")
    return ax

print("✓ Styling loaded  |  TF", tf.__version__)

TensorFlow 2.19.0  |  Keras 3.10.0
✓ Styling loaded  |  TF 2.19.0


In [8]:
def load_split(tag):
    d = np.load(f"{WORKING_PATH1}/features_{tag}.npz", allow_pickle=True)
    X       = d["X"].astype(np.float32)
    y       = d["y_enc"].astype(np.int64)      # integer-encoded labels
    classes = d["classes"]                      # ['balastic' 'footsteps' 'noise']
    return X, y, classes

X_train, y_train, label_names = load_split("train")
X_val,   y_val,   _           = load_split("val")
X_test,  y_test,  _           = load_split("test")

N_CLASSES  = len(label_names)
N_FEATURES = X_train.shape[1]

CLASS_NAMES  = list(label_names)                # ['balastic', 'footsteps', 'noise']
CLASS_COLORS = {"balastic": "#2196F3", "footsteps": "#FF9800", "noise": "#4CAF50"}
PALETTE      = [CLASS_COLORS[c] for c in CLASS_NAMES]

print(f"Feature dim  : {N_FEATURES}")
print(f"Classes      : {CLASS_NAMES}  ({N_CLASSES})")
print(f"Train        : {X_train.shape}   labels={np.bincount(y_train)}")
print(f"Val          : {X_val.shape}     labels={np.bincount(y_val)}")
print(f"Test         : {X_test.shape}    labels={np.bincount(y_test)}")

# One-hot encode for Keras
Y_train = to_categorical(y_train, N_CLASSES)
Y_val   = to_categorical(y_val,   N_CLASSES)
Y_test  = to_categorical(y_test,  N_CLASSES)

print("\n✓ All splits loaded successfully")

Feature dim  : 24
Classes      : ['balastic', 'footsteps', 'noise']  (3)
Train        : (775, 24)   labels=[261 296 218]
Val          : (99, 24)     labels=[56 12 31]
Test         : (100, 24)    labels=[57 12 31]

✓ All splits loaded successfully


## Stage 1 — Feature Distribution & Class Balance Audit

Before training, confirm that normalised features are well-conditioned and
classes are balanced across all splits.

In [9]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Feature Space Audit — Pre-Training Verification",
             fontsize=14, fontweight="bold", y=1.01)

# (a) Per-class feature mean ± std heatmap
ax = axes[0, 0]
means = np.array([X_train[y_train == i].mean(axis=0) for i in range(N_CLASSES)])
im = ax.imshow(means, aspect="auto", cmap="RdYlBu_r")
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Feature Index"); ax.set_title("(a) Per-Class Feature Means")
plt.colorbar(im, ax=ax, shrink=0.85)
paper_axes(ax)

# (b) Feature variance across train set
ax = axes[0, 1]
vars_ = X_train.var(axis=0)
ax.bar(range(N_FEATURES), vars_, color="#5C6BC0", alpha=0.8, edgecolor="none")
ax.set_xlabel("Feature Index"); ax.set_ylabel("Variance")
ax.set_title("(b) Feature Variance (Train)")
paper_axes(ax)

# (c) Class counts per split
ax = axes[1, 0]
split_names = ["Train", "Val", "Test"]
ys = [y_train, y_val, y_test]
x = np.arange(N_CLASSES); width = 0.25
for si, (sn, y_s) in enumerate(zip(split_names, ys)):
    counts = np.bincount(y_s, minlength=N_CLASSES)
    ax.bar(x + si*width, counts, width, label=sn,
           color=["#42A5F5","#FFA726","#66BB6A"][si], alpha=0.85, edgecolor="none")
ax.set_xticks(x + width); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel("Count"); ax.set_title("(c) Class Distribution per Split")
ax.legend(); paper_axes(ax)

# (d) PairPlot proxy — scatter first 3 MFCCs coloured by class
ax = axes[1, 1]
for i, cn in enumerate(CLASS_NAMES):
    idx = y_train == i
    ax.scatter(X_train[idx, 0], X_train[idx, 1],
               c=PALETTE[i], label=cn, alpha=0.5, s=18, edgecolors="none")
ax.set_xlabel("Feature 0 (MFCC-1 mean)")
ax.set_ylabel("Feature 1 (MFCC-2 mean)")
ax.set_title("(d) Feature 0 vs 1 — Train")
ax.legend(); paper_axes(ax)

plt.tight_layout()
plt.savefig(f"{FIGURES_PATH}/fig_feature_audit.pdf")
plt.show()
print("Saved → fig_feature_audit.pdf")

Saved → fig_feature_audit.pdf


## Stage 2 — Model Architecture Design

Three architectures are trained and compared under TinyML constraints:

| Model | Architecture | Target use |
|---|---|---|
| **MLP-Tiny** | 3-layer MLP (64→32→16) | Baseline, smallest footprint |
| **MLP-Deep** | 5-layer MLP with BatchNorm + Dropout | Best accuracy |
| **MLP-BN** | 3-layer MLP + BatchNorm (no Dropout) | Fast inference |

All models use **24-dim** input → **3-class** softmax output.  
Post-training **INT8 quantisation** targets ESP32 SRAM < 50 KB.

In [10]:
def build_mlp_tiny(input_dim=24, n_classes=3):
    inp = keras.Input(shape=(input_dim,), name="input")
    x = layers.Dense(64, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4))(inp)
    x = layers.Dense(32, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax", name="output")(x)
    model = keras.Model(inp, out, name="MLP_Tiny")
    return model

def build_mlp_deep(input_dim=24, n_classes=3):
    inp = keras.Input(shape=(input_dim,), name="input")
    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(inp)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax", name="output")(x)
    model = keras.Model(inp, out, name="MLP_Deep")
    return model

def build_mlp_bn(input_dim=24, n_classes=3):
    inp = keras.Input(shape=(input_dim,), name="input")
    x = layers.Dense(64)(inp)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dense(32)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax", name="output")(x)
    model = keras.Model(inp, out, name="MLP_BN")
    return model

# Summarise all three
for build_fn in [build_mlp_tiny, build_mlp_deep, build_mlp_bn]:
    m = build_fn()
    params = m.count_params()
    print(f"{m.name:<12}  params={params:,}  est. size={params*4/1024:.1f} KB (float32)")

MLP_Tiny      params=4,259  est. size=16.6 KB (float32)
MLP_Deep      params=14,883  est. size=58.1 KB (float32)
MLP_BN        params=4,643  est. size=18.1 KB (float32)


In [11]:
EPOCHS    = 120
BATCH     = 32
LR        = 1e-3

histories = {}
trained   = {}

for build_fn in [build_mlp_tiny, build_mlp_deep, build_mlp_bn]:
    model = build_fn()
    model.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    cb_list = [
        callbacks.EarlyStopping(monitor="val_accuracy", patience=18,
                                restore_best_weights=True, verbose=0),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                    patience=8, min_lr=1e-6, verbose=0),
        callbacks.ModelCheckpoint(
            f"{WORKING_PATH}/{model.name}_best.h5",
            monitor="val_accuracy", save_best_only=True, verbose=0)
    ]

    t0  = time.time()
    hist = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=EPOCHS, batch_size=BATCH,
        callbacks=cb_list, verbose=0
    )
    elapsed = time.time() - t0

    val_acc  = max(hist.history["val_accuracy"])
    test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)

    histories[model.name] = hist.history
    trained[model.name]   = model

    print(f"[{model.name}]  val_acc={val_acc:.4f}  test_acc={test_acc:.4f}"
          f"  epochs={len(hist.history['loss'])}  time={elapsed:.1f}s")

[MLP_Tiny]  val_acc=0.8788  test_acc=0.8000  epochs=38  time=8.3s


[MLP_Deep]  val_acc=0.8687  test_acc=0.8000  epochs=44  time=14.0s


[MLP_BN]  val_acc=0.9091  test_acc=0.8200  epochs=42  time=10.1s


In [13]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Training Dynamics — All Models", fontsize=14, fontweight="bold")

for col, mname in enumerate(trained.keys()):
    h = histories[mname]

    # Accuracy
    ax = axes[0, col]
    ax.plot(h["accuracy"],     color="#1976D2", label="Train")
    ax.plot(h["val_accuracy"], color="#F57C00", label="Val", linestyle="--")
    best_ep = int(np.argmax(h["val_accuracy"]))
    ax
